In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.fft import dst, idst
from scipy.linalg import eigh_tridiagonal
from scipy.optimize import least_squares

from shared.common import load_config, normal_density, split_point
from shared.helper_funcs import rotate, rural_widths

In [ ]:
config = load_config()['field']
inp, out = Path('../input'), Path('../output')
log_scale = config['log_scale_mg_m3']
width_bounds = config['width_scale_bounds']
angle_bound = config['bearing_offset_bound_deg']
deposition = config['deposition_m_s']
high_deposition = config['deposition_high_m_s']
ca_models = [
    ('ca_flat', False, 0.0), ('ca_shear', True, 0.0), ('ca_dep', False, deposition),
    ('ca_shear_dep', True, deposition), ('ca_shear_high_dep', True, high_deposition),
]

In [ ]:
runs = pd.read_csv(inp / 'runs.csv')
receptors = pd.read_csv(inp / 'receptors.csv')
profiles = pd.read_csv(inp / 'profiles.csv')
merged = receptors.merge(runs[['run', 'u2_m_s', 'toward_deg', 'q_g_s', 'h_m', 'stability', 'sigma_theta_deg']],
                        on='run', validate='many_to_one')

In [ ]:
def width_sq(x, stability, theta_sd, scales, use_met):
    sigma_y, sigma_z = rural_widths(x, stability)
    if use_met:
        # angular-meander proxy + rural width floor
        sigma_y = np.hypot(config['angle_floor_fraction']*sigma_y, x*np.deg2rad(theta_sd))
    return (scales[0]*sigma_y)**2, (scales[1]*sigma_z)**2

def gaussian(inputs, parameters, use_met):
    east, north, z, theta, q, h, u2, stability, theta_sd = inputs
    x, y = rotate(east, north, theta + parameters[2])
    sy2, sz2 = width_sq(np.maximum(x, 1e-6), stability, theta_sd, parameters[:2], use_met)
    vertical = normal_density(z, h, np.sqrt(sz2)) + normal_density(z, -h, np.sqrt(sz2))
    return np.where(x > 0, 1000*q/u2*normal_density(y, 0, np.sqrt(sy2))*vertical, 0)

def gaussian_inputs(frame):
    cols = ['east_m', 'north_m', 'z_m', 'toward_deg', 'q_g_s', 'h_m', 'u2_m_s', 'stability', 'sigma_theta_deg']
    return [frame[col].to_numpy() for col in cols]

In [ ]:
# train on arc receptors only; equal dates, runs within dates, arcs within runs
fold_parameters, fit_rows = {}, []
arc = merged.query("array == 'arc'")
initial = np.array([1.0, 1.0, 0.0])
bounds = ([width_bounds[0], width_bounds[0], -angle_bound], [width_bounds[1], width_bounds[1], angle_bound])
for fold in sorted(runs['fold'].unique()):
    train = arc.loc[arc['fold'].ne(fold)].copy()
    n_days = train['met_day'].nunique()
    n_runs = train.groupby('met_day')['run'].transform('nunique')
    n_arcs = train.groupby('run')['radius_m'].transform('nunique')
    n_rows = train.groupby(['run', 'radius_m'])['obs_id'].transform('size')
    weight = np.sqrt(1/(n_days*n_runs*n_arcs*n_rows))
    target = np.log1p(train['observed_mg_m3'].to_numpy()/log_scale)
    inputs = gaussian_inputs(train)

    starts = [initial, [1.8, 1, 8], [1.8, 1, -8], [2.7, 0.7, 0], [0.8, 0.7, 15]]
    for label, use_met in [('briggs', False), ('met', True)]:
        def residual(par):
            return weight*(np.log1p(gaussian(inputs, par, use_met)/log_scale) - target)

        candidates = [least_squares(residual, start, bounds=bounds, x_scale=[1, 1, 12], max_nfev=160)
                      for start in starts]
        result = min((fit for fit in candidates if fit.success), key=lambda fit: fit.cost)
        fold_parameters[(fold, label)] = result.x
        fit_rows.append((fold, label, *result.x, n_days, train['run'].nunique(),
                         np.sqrt(2*result.cost), np.any(result.active_mask != 0)))
fits = pd.DataFrame(fit_rows, columns=['held_out_fold', 'closure', 'scale_y', 'scale_z', 'offset_deg',
                                       'training_days', 'training_runs', 'train_rmse_log1p', 'at_bound'])
fits.to_csv(out / 'fit_parameters.csv', index=False)
display(fits.round(4))

In [ ]:
def vertical_grid(top, refine):
    dz = config['dz_surface_m']/refine
    growth = 1 + (config['vertical_growth'] - 1)/refine
    cap = config['dz_cap_m']/refine
    edges = [0.0]
    while edges[-1] < top:
        edges.append(edges[-1] + dz)
        dz = min(cap, dz*growth)
    edges = np.asarray(edges)
    return (edges[:-1] + edges[1:])/2, np.diff(edges)

def local_speed(z, reference, profile, shear):
    if not shear:
        return np.full_like(z, reference)
    h = np.log(profile['height_m'].to_numpy())
    u = profile['speed_m_s'].to_numpy()
    return reference*np.interp(np.log(z), h, u)/np.interp(np.log(2), h, u)

def z_operator(z, dz, speed):
    # reversible nearest-neighbor generator after division by sqrt(U dz)
    pi = speed*dz
    faces = 1/np.diff(z)
    diagonal = -(np.r_[0, faces] + np.r_[faces, 1/dz[-1]])/pi
    off_diag = faces/np.sqrt(pi[:-1]*pi[1:])
    decay, basis = eigh_tridiagonal(diagonal, off_diag)
    return decay, basis, np.sqrt(pi)

In [ ]:
def lattice_predict(samples, parameters, wind_profile, shear, v_dep, refine=1):
    # directed x march for flux, with local y/z jumps in continuous distance
    row = samples.iloc[0]
    x, y = rotate(samples['east_m'].to_numpy(), samples['north_m'].to_numpy(), row.toward_deg + parameters[2])
    z = samples['z_m'].to_numpy()
    downwind = x > 0
    query = np.flatnonzero(downwind)
    xq, group = np.unique(x[query], return_inverse=True)
    end = xq[-1]
    dx = config['dx_m']/refine
    near_y = np.sqrt(width_sq(50, row.stability, row.sigma_theta_deg, parameters, True)[0])
    dy = max(config['dy_m'], min(12.0, near_y/8))/refine
    sd_y, sd_z = np.sqrt(width_sq(end, row.stability, row.sigma_theta_deg, parameters, True))
    # choose domain from modeled spread and receptor coordinates, never observations
    ratio = np.sqrt(row.u2_m_s/local_speed(np.array([0.001]), row.u2_m_s, wind_profile, shear)[0])
    n_y = int(np.ceil(max(abs(y[query]).max() + 2*dy, config['domain_sigmas']*sd_y*max(1, ratio))/dy))
    yg = np.arange(-n_y, n_y+1)*dy
    zg, dz = vertical_grid(max(z.max() + 4, row.h_m + config['domain_sigmas']*sd_z*max(1, ratio)), refine)
    wind = local_speed(zg, row.u2_m_s, wind_profile, shear)
    decay_z, basis, root_pi = z_operator(zg, dz, wind)
    decay_y = -4*np.sin(np.pi*np.arange(1, len(yg)+1)/(2*(len(yg)+1)))**2/dy**2
    y_rate = decay_y[:, None]/wind[None, :]
    y_weights = dst(np.ones(len(yg)), type=1, norm='ortho')
    delta = np.zeros(len(yg))
    delta[n_y] = 1
    vertical = np.zeros(len(zg))
    emitted = 1000*row.q_g_s
    for level, weight in split_point(zg, row.h_m):
        vertical[level] = emitted*weight
    state = dst(delta, type=1, norm='ortho')[:, None]*(vertical/root_pi)[None, :]

    # plane coordinates include each receptor x, plus subdivisions for splitting
    planes = np.unique(np.r_[0.0, np.arange(dx, end, dx), xq])
    plane_idx = np.searchsorted(planes, xq)
    packed = pd.Series(query).groupby(plane_idx[group]).agg(list).to_dict()
    vy, vz = width_sq(planes, row.stability, row.sigma_theta_deg, parameters, True)
    y_increments = 0.5*row.u2_m_s*np.diff(vy)
    z_increments = 0.5*row.u2_m_s*np.diff(vz)
    prediction = np.zeros(len(samples))
    removed, lateral, upper = 0.0, 0.0, 0.0
    remaining = emitted
    neg_tol = 0.0

    for j, (distance, delta_y, delta_z) in enumerate(zip(np.diff(planes), y_increments, z_increments), start=1):
        # symmetric splitting; exact exponential of each local jump operator
        sink = np.exp(-v_dep*distance/(2*wind[0]*dz[0]))
        y_half = np.exp(0.5*delta_y*y_rate)
        removed += root_pi[0]*(y_weights @ state[:, 0])*(1-sink)
        state[:, 0] *= sink
        after_dep = (y_weights @ state) @ root_pi
        state *= y_half
        after_y = (y_weights @ state) @ root_pi
        lateral += after_dep-after_y
        state = ((state @ basis)*np.exp(delta_z*decay_z)) @ basis.T
        after_z = (y_weights @ state) @ root_pi
        upper += after_y-after_z
        state *= y_half
        after_y = (y_weights @ state) @ root_pi
        lateral += after_z-after_y
        removed += root_pi[0]*(y_weights @ state[:, 0])*(1-sink)
        state[:, 0] *= sink
        remaining = (y_weights @ state) @ root_pi

        ids = packed.get(j)
        if ids is not None:
            concentration = idst(state, type=1, norm='ortho', axis=0)/(dy*root_pi[None, :])
            for k in ids:
                prediction[k] = sum(wy*wz*concentration[iy, iz]
                                    for iy, wy in split_point(yg, y[k])
                                    for iz, wz in split_point(zg, z[k]))
            neg_tol = min(neg_tol, prediction[ids].min())

    audit = dict(end_x_m=end, remaining_frac=remaining/emitted, deposited_frac=removed/emitted,
                 lateral_frac=lateral/emitted, upper_frac=upper/emitted,
                 balance_residual_frac=(remaining + removed + lateral + upper)/emitted - 1,
                 min_receptor_mg_m3=neg_tol, dy_m=dy, y_cells=len(yg), z_cells=len(zg), x_steps=len(planes)-1)
    return np.maximum(0, prediction), audit

In [ ]:
# class-stratified, input-only representative releases for grid refinement
refine_runs = set(runs.sort_values('u2_m_s').groupby('stability')['run'].agg(lambda group: group.iloc[len(group)//2]))
outputs, audits, refinements = [], [], []
for run, samples in merged.groupby('run', sort=True):
    fold = samples['fold'].iat[0]
    met_params = fold_parameters[(fold, 'met')]
    briggs_params = fold_parameters[(fold, 'briggs')]
    inputs = gaussian_inputs(samples)
    pred = samples[['obs_id', 'run', 'fold', 'met_day', 'array', 'radius_m', 'bearing_deg', 'z_m',
                    'observed_mg_m3', 'arc_factor']].copy()
    pred['gaussian_briggs_raw'] = gaussian(inputs, initial, False)
    pred['gaussian_briggs'] = gaussian(inputs, briggs_params, False)
    pred['gaussian_met'] = gaussian(inputs, met_params, True)
    wind = profiles.query('run == @run').sort_values('height_m')

    for model, shear, v_dep in ca_models:
        pred[model], audit = lattice_predict(samples, met_params, wind, shear, v_dep)
        audits.append(dict(run=run, model=model, **audit))
        if run in refine_runs and model in ('ca_flat', 'ca_shear_dep'):
            finer, _ = lattice_predict(samples, met_params, wind, shear, v_dep, refine=2)
            refinements.append(pred[['obs_id', 'run', 'array', 'observed_mg_m3']].assign(
                model=model, coarse_mg_m3=pred[model], finer_mg_m3=finer))
    outputs.append(pred)
predictions = pd.concat(outputs, ignore_index=True)
audit = pd.DataFrame(audits)
refinement = pd.concat(refinements, ignore_index=True)

In [ ]:
# same U/K with vs without a positive sink: pointwise ordering, up to roundoff
tol = 1e-7
assert (predictions['ca_dep'] <= predictions['ca_flat'] + tol).all()
assert (predictions['ca_shear_high_dep'] <= predictions['ca_shear_dep'] + tol).all()
assert (predictions['ca_shear_dep'] <= predictions['ca_shear'] + tol).all()
assert audit['balance_residual_frac'].abs().max() < 1e-9
assert audit['min_receptor_mg_m3'].min() > -tol
predictions.to_csv(out / 'predictions.csv', index=False)
audit.to_csv(out / 'field_mass_balance.csv', index=False)
refinement.to_csv(out / 'refinement.csv', index=False)
display(audit.groupby('model', as_index=False).agg(
    mean_deposited=('deposited_frac', 'mean'), max_top_loss=('upper_frac', 'max'),
    max_side_loss=('lateral_frac', 'max'), max_balance_error=('balance_residual_frac', lambda x: x.abs().max())))